In [3]:
from pathlib import Path
from datetime import datetime
import time
import sys

import nbformat
from nbclient import NotebookClient

# ============================================================
# CONFIGURAÇÃO DOS DIRETÓRIOS
# ============================================================

# O Jupyter está sendo executado dentro de OFI/src
DIRETORIO_ATUAL = Path.cwd()

# Como estamos em OFI/src,
# o diretório pai é a raiz da OFI.
BASE = DIRETORIO_ATUAL.parent

# Pasta dos notebooks
SRC = BASE / "src"


# ============================================================
# CONFIGURAÇÃO DO PYTHON PATH
# ============================================================

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


print(f"Diretório atual: {DIRETORIO_ATUAL}")
print(f"BASE OFI:        {BASE}")
print(f"SRC:             {SRC}")


# Sequência oficial de processamento da OFI
NOTEBOOKS = [
    "extrair_dados.ipynb",
    "Gerar_arquivos_D2.ipynb",
    "Atualizar_java.script_home_page.ipynb",
    
    "grafico_status_navegacao.ipynb",
    "campo_atendimento.ipynb",
    "grafico_gantt.ipynb",
    "trajetoria_acumulada.ipynb",
    "grafico_deslocamento_intervalo.ipynb",
    "gerar_dashboard.ipynb",
]

# True  -> para no primeiro erro
# False -> continua mesmo se um notebook falhar
PARAR_SE_ERRO = True


# ============================================================
# FORMATAÇÃO DE TEMPO
# ============================================================

def formatar_tempo(segundos):

    if segundos < 60:
        return f"{segundos:.2f} s"

    minutos = int(segundos // 60)
    segundos_restantes = segundos % 60

    if minutos < 60:
        return f"{minutos} min {segundos_restantes:.2f} s"

    horas = int(minutos // 60)
    minutos_restantes = minutos % 60

    return (
        f"{horas} h "
        f"{minutos_restantes} min "
        f"{segundos_restantes:.2f} s"
    )


# ============================================================
# EXECUÇÃO DE NOTEBOOK
# ============================================================

def executar_notebook(caminho):

    with open(caminho, "r", encoding="utf-8") as arquivo:

        notebook = nbformat.read(
            arquivo,
            as_version=4
        )

    # ========================================================
    # CONFIGURAÇÃO TEMPORÁRIA DO KERNEL
    # ========================================================

    codigo_configuracao = f"""
import sys
import os
from pathlib import Path

BASE = Path(r"{BASE}")
SRC = Path(r"{SRC}")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Caminho do PROJ dentro do ambiente geo_video
PROJ_DIR = Path(sys.prefix) / "Library" / "share" / "proj"

os.environ["PROJ_DATA"] = str(PROJ_DIR)
os.environ["PROJ_LIB"] = str(PROJ_DIR)

import pyproj

pyproj.datadir.set_data_dir(str(PROJ_DIR))
"""

    celula_configuracao = nbformat.v4.new_code_cell(
        codigo_configuracao
    )

    # Insere essa célula no início do notebook
    # somente na memória
    notebook.cells.insert(
        0,
        celula_configuracao
    )

    # ========================================================
    # NOTEBOOKCLIENT
    # ========================================================

    cliente = NotebookClient(
        notebook,
        timeout=None,
        kernel_name="geo_video",
        allow_errors=False,
    )

    # Executa o notebook tendo a raiz OFI como diretório
    cliente.execute(
        cwd=str(BASE)
    )


# ============================================================
# INÍCIO
# ============================================================

inicio_total = time.perf_counter()
inicio_relogio = datetime.now()

print()
print("=" * 75)
print("                    OFI - CÓDIGO MESTRE")
print("=" * 75)

print(
    f"Início: "
    f"{inicio_relogio.strftime('%d/%m/%Y %H:%M:%S')}"
)

print(f"BASE: {BASE}")
print(f"SRC:  {SRC}")

print()
print(f"Total de notebooks: {len(NOTEBOOKS)}")

print()
print("SEQUÊNCIA DE EXECUÇÃO")
print("-" * 75)

for numero, notebook in enumerate(NOTEBOOKS, start=1):

    print(
        f"{numero:02d}. {notebook}"
    )


# ============================================================
# EXECUÇÃO DA SEQUÊNCIA
# ============================================================

resultados = []

for numero, nome_notebook in enumerate(
    NOTEBOOKS,
    start=1
):

    caminho = SRC / nome_notebook

    print()
    print("=" * 75)
    print(
        f"[{numero:02d}/{len(NOTEBOOKS):02d}] "
        f"EXECUTANDO"
    )
    print("=" * 75)

    print(f"Notebook: {nome_notebook}")

    # --------------------------------------------------------
    # VERIFICA SE O ARQUIVO EXISTE
    # --------------------------------------------------------

    if not caminho.exists():

        print()
        print("✗ ERRO")
        print("Notebook não encontrado:")
        print(caminho)

        resultados.append({
            "notebook": nome_notebook,
            "tempo": 0,
            "status": "ERRO - ARQUIVO NÃO ENCONTRADO",
        })

        if PARAR_SE_ERRO:
            break

        continue

    # --------------------------------------------------------
    # CRONÔMETRO INDIVIDUAL
    # --------------------------------------------------------

    inicio_notebook = time.perf_counter()

    try:

        executar_notebook(caminho)

        fim_notebook = time.perf_counter()

        tempo = fim_notebook - inicio_notebook

        resultados.append({
            "notebook": nome_notebook,
            "tempo": tempo,
            "status": "OK",
        })

        print()
        print("✓ CONCLUÍDO")
        print(
            f"Tempo: {formatar_tempo(tempo)}"
        )

    except Exception as erro:

        fim_notebook = time.perf_counter()

        tempo = fim_notebook - inicio_notebook

        resultados.append({
            "notebook": nome_notebook,
            "tempo": tempo,
            "status": "ERRO",
        })

        print()
        print("✗ ERRO")
        print(
            f"Tempo até o erro: "
            f"{formatar_tempo(tempo)}"
        )

        print()
        print("Detalhes do erro:")
        print(erro)

        if PARAR_SE_ERRO:

            print()
            print(
                "EXECUÇÃO INTERROMPIDA."
            )

            break


# ============================================================
# RESUMO FINAL
# ============================================================

fim_total = time.perf_counter()

tempo_total = fim_total - inicio_total

print()
print()
print("=" * 75)
print("                    RESUMO DA EXECUÇÃO")
print("=" * 75)

print()

for numero, resultado in enumerate(
    resultados,
    start=1
):

    if resultado["status"] == "OK":

        simbolo = "✓"

    else:

        simbolo = "✗"

    print(
        f"{numero:02d}. "
        f"{simbolo} "
        f"{resultado['notebook']:<45} "
        f"{formatar_tempo(resultado['tempo'])}"
    )


# ============================================================
# ESTATÍSTICAS
# ============================================================

sucessos = sum(
    resultado["status"] == "OK"
    for resultado in resultados
)

erros = sum(
    resultado["status"] != "OK"
    for resultado in resultados
)

print()
print("-" * 75)

print(
    f"Total previsto:      {len(NOTEBOOKS)}"
)

print(
    f"Executados:          {len(resultados)}"
)

print(
    f"Sucessos:            {sucessos}"
)

print(
    f"Erros:               {erros}"
)

print()
print(
    f"TEMPO TOTAL: "
    f"{formatar_tempo(tempo_total)}"
)

fim_relogio = datetime.now()

print(
    f"Fim: "
    f"{fim_relogio.strftime('%d/%m/%Y %H:%M:%S')}"
)

print()
print("=" * 75)

if (
    erros == 0
    and len(resultados) == len(NOTEBOOKS)
):

    print(
        "✓ OFI PROCESSADA COM SUCESSO"
    )

elif erros > 0:

    print(
        "✗ OFI PROCESSADA COM ERRO"
    )

else:

    print(
        "⚠ OFI PROCESSADA PARCIALMENTE"
    )

print("=" * 75)


Diretório atual: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\src
BASE OFI:        C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi
SRC:             C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\src

                    OFI - CÓDIGO MESTRE
Início: 28/08/2026 17:34:52
BASE: C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi
SRC:  C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\src

Total de notebooks: 9

SEQUÊNCIA DE EXECUÇÃO
---------------------------------------------------------------------------
01. extrair_dados.ipynb
02. Gerar_arquivos_D2.ipynb
03. Atualizar_java.script_home_page.ipynb
04. grafico_status_navegacao.ipynb
05. campo_atendimento.ipynb
06. grafico_gantt.ipynb
07. trajetoria_acumulada.ipynb
08. grafico_deslocamento_intervalo.ipynb
09. gerar_dashboard.i

In [9]:
import geopandas
import fiona

print("Python:", sys.executable)
print("GeoPandas:", geopandas.__version__)
print("Fiona:", fiona.__version__)

Python: C:\Users\roger\Anaconda3\python.exe
GeoPandas: 0.13.2
Fiona: 1.10.1
